# Amazon Fine Food Reviews — LSTM Sentiment Classification
Dataset: [Kaggle Amazon Fine Food Reviews](https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews).

Binary classification: scores 1–2 = negative (0), scores 4–5 = positive (1). Neutral score 3 is excluded. Download and extract `Reviews.csv` into the notebook directory (or update the path below).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

## 1. Load and sample the dataset

In [ ]:
df = pd.read_csv("Reviews.csv", usecols=["Text", "Score"])
df = df.dropna().drop_duplicates(subset="Text")
df = df[df["Score"] != 3].sample(n=5000, random_state=42).copy()
df["Sentiment"] = (df["Score"] >= 4).astype(int)
print(df["Sentiment"].value_counts())
df.head()

## 2. Split into training, validation and test sets

In [ ]:
X = df["Text"]
y = df["Sentiment"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

## 3. Tokenize and pad sequences
Fit the tokenizer only on training text to avoid data leakage. Use a 5,000-word vocabulary and sequence length of 50.

In [ ]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=50)
X_val = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=50)
X_test = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=50)
print(X_train.shape, X_val.shape, X_test.shape)

## 4. Build the exact LSTM architecture

In [ ]:
model = Sequential([
    Embedding(5000, 32),
    LSTM(32),
    Dense(1, activation="sigmoid")
])
model.build(input_shape=(None, 50))
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

## 5. Train for 5 epochs

In [ ]:
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_val, y_val))

## 6. Evaluate on unseen test data

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Majority-class baseline accuracy: {y_test.value_counts(normalize=True).max():.4f}")

## 7. Training vs. validation curves

In [ ]:
epochs = range(1, 6)
plt.plot(epochs, history.history["accuracy"], label="Training accuracy")
plt.plot(epochs, history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs. Validation Accuracy")
plt.xticks(list(epochs))
plt.legend()
plt.show()

In [ ]:
plt.plot(epochs, history.history["loss"], label="Training loss")
plt.plot(epochs, history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs. Validation Loss")
plt.xticks(list(epochs))
plt.legend()
plt.show()

## 8. Forget gate explanation
The forget gate allows an LSTM to selectively retain or discard information from its previous cell state. Unlike a plain RNN, this gated memory helps preserve important information over longer sequences and reduces the vanishing-gradient problem.